In [1]:
# Install required libraries (Colab safe)
!pip install -q datasets transformers matplotlib numpy pandas

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

In [2]:
from datasets import load_dataset

# Load official dataset (27 emotions + neutral)
dataset = load_dataset("google-research-datasets/go_emotions", "simplified")

print(dataset)

README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})


In [3]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/sivachandrasajeev/augmentation/training_log.csv
/kaggle/input/datasets/sivachandrasajeev/augmentation/test_emotion_labels.npy
/kaggle/input/datasets/sivachandrasajeev/augmentation/aug_stats.json
/kaggle/input/datasets/sivachandrasajeev/augmentation/test_sentiment_logits.npy
/kaggle/input/datasets/sivachandrasajeev/augmentation/training_args.bin
/kaggle/input/datasets/sivachandrasajeev/augmentation/training_args.json
/kaggle/input/datasets/sivachandrasajeev/augmentation/tokenizer.json
/kaggle/input/datasets/sivachandrasajeev/augmentation/test_sentiment_labels.npy
/kaggle/input/datasets/sivachandrasajeev/augmentation/tokenizer_config.json
/kaggle/input/datasets/sivachandrasajeev/augmentation/val_emotion_logits.npy
/kaggle/input/datasets/sivachandrasajeev/augmentation/val_emotion_labels.npy
/kaggle/input/datasets/sivachandrasajeev/augmentation/emotion_names.json
/kaggle/input/datasets/sivachandrasajeev/augmentation/val_emotion_probs.npy
/kaggle/input/datasets/sivach

In [4]:
# Extract label names directly from dataset
EMOTION_NAMES = dataset["train"].features["labels"].feature.names

print("Number of labels:", len(EMOTION_NAMES))
print("Labels:", EMOTION_NAMES)

Number of labels: 28
Labels: ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


In [5]:
import numpy as np

def encode_labels(example):
    multi = np.zeros(len(EMOTION_NAMES))
    for idx in example["labels"]:
        multi[idx] = 1
    example["multi_labels"] = multi
    return example

dataset = dataset.map(encode_labels)

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [6]:
def analyze_neutral_rows(dataset_split):

    neutral_only = 0
    neutral_plus_other = 0
    no_neutral = 0

    for labels in dataset_split["labels"]:

        if 27 in labels:
            if len(labels) == 1:
                neutral_only += 1
            else:
                neutral_plus_other += 1
        else:
            no_neutral += 1

    total = len(dataset_split)

    print(f"Total samples: {total}")
    print(f"Neutral only rows: {neutral_only}")
    print(f"Neutral + other emotion rows: {neutral_plus_other}")
    print(f"No neutral rows: {no_neutral}")
    print("-" * 40)

    return neutral_only, neutral_plus_other, no_neutral

In [7]:
print("TRAIN SPLIT")
train_counts = analyze_neutral_rows(dataset["train"])

print("VALIDATION SPLIT")
val_counts = analyze_neutral_rows(dataset["validation"])

print("TEST SPLIT")
test_counts = analyze_neutral_rows(dataset["test"])

TRAIN SPLIT
Total samples: 43410
Neutral only rows: 12823
Neutral + other emotion rows: 1396
No neutral rows: 29191
----------------------------------------
VALIDATION SPLIT
Total samples: 5426
Neutral only rows: 1592
Neutral + other emotion rows: 174
No neutral rows: 3660
----------------------------------------
TEST SPLIT
Total samples: 5427
Neutral only rows: 1606
Neutral + other emotion rows: 181
No neutral rows: 3640
----------------------------------------


In [8]:
def remove_neutral_only(example):
    return not (27 in example["labels"] and len(example["labels"]) == 1)

dataset["train"] = dataset["train"].filter(remove_neutral_only)
dataset["validation"] = dataset["validation"].filter(remove_neutral_only)
dataset["test"] = dataset["test"].filter(remove_neutral_only)

Filter:   0%|          | 0/43410 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5426 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [9]:
def strip_neutral(example):
    example["labels"] = [l for l in example["labels"] if l != 27]
    return example

dataset["train"] = dataset["train"].map(strip_neutral)
dataset["validation"] = dataset["validation"].map(strip_neutral)
dataset["test"] = dataset["test"].map(strip_neutral)

Map:   0%|          | 0/30587 [00:00<?, ? examples/s]

Map:   0%|          | 0/3834 [00:00<?, ? examples/s]

Map:   0%|          | 0/3821 [00:00<?, ? examples/s]

In [10]:
print("Train rows:", len(dataset["train"]))
print("Validation rows:", len(dataset["validation"]))
print("Test rows:", len(dataset["test"]))

Train rows: 30587
Validation rows: 3834
Test rows: 3821


In [11]:
import pandas as pd

In [12]:
EMOTION_NAMES_NO_NEUTRAL = [
    'admiration','amusement','anger','annoyance','approval','caring',
    'confusion','curiosity','desire','disappointment','disapproval',
    'disgust','embarrassment','excitement','fear','gratitude','grief',
    'joy','love','nervousness','optimism','pride','realization',
    'relief','remorse','sadness','surprise'
]

In [13]:
labels = dataset["train"]["labels"]

multi_hot = np.zeros((len(labels), 27))

for i, row in enumerate(labels):
    for label in row:
        multi_hot[i, label] = 1

emotion_counts = multi_hot.sum(axis=0)

distribution_df = pd.DataFrame({
    "Emotion": EMOTION_NAMES_NO_NEUTRAL,
    "Count": emotion_counts
}).sort_values(by="Count", ascending=False)

distribution_df

,Emotion,Count
0,admiration,4130.0
4,approval,2939.0
15,gratitude,2662.0
3,annoyance,2470.0
1,amusement,2328.0
7,curiosity,2191.0
18,love,2086.0
10,disapproval,2022.0
20,optimism,1581.0
2,anger,1567.0


In [14]:
def count_label_types(split, split_name):

    single = 0
    multi = 0

    for row in split["labels"]:
        if len(row) == 1:
            single += 1
        elif len(row) > 1:
            multi += 1

    total = len(split["labels"])

    print(f"\n===== {split_name.upper()} =====")
    print("Total samples:", total)
    print("Single-label samples:", single)
    print("Multi-label samples:", multi)
    print("Single %: {:.2f}%".format(100 * single / total))
    print("Multi %: {:.2f}%".format(100 * multi / total))


count_label_types(dataset["train"], "train")
count_label_types(dataset["validation"], "validation")
count_label_types(dataset["test"], "test")


===== TRAIN =====
Total samples: 30587
Single-label samples: 24820
Multi-label samples: 5767
Single %: 81.15%
Multi %: 18.85%

===== VALIDATION =====
Total samples: 3834
Single-label samples: 3122
Multi-label samples: 712
Single %: 81.43%
Multi %: 18.57%

===== TEST =====
Total samples: 3821
Single-label samples: 3154
Multi-label samples: 667
Single %: 82.54%
Multi %: 17.46%


In [15]:
def neutral_sanity_check(split, split_name):

    neutral_only = 0
    neutral_mixed = 0
    total = len(split["labels"])

    for row in split["labels"]:

        if 27 in row:
            if len(row) == 1:
                neutral_only += 1
            else:
                neutral_mixed += 1

    print(f"\n===== {split_name.upper()} =====")
    print("Total samples:", total)
    print("Neutral only:", neutral_only)
    print("Neutral + other emotion:", neutral_mixed)


neutral_sanity_check(dataset["train"], "train")
neutral_sanity_check(dataset["validation"], "validation")
neutral_sanity_check(dataset["test"], "test")


===== TRAIN =====
Total samples: 30587
Neutral only: 0
Neutral + other emotion: 0

===== VALIDATION =====
Total samples: 3834
Neutral only: 0
Neutral + other emotion: 0

===== TEST =====
Total samples: 3821
Neutral only: 0
Neutral + other emotion: 0


In [16]:
import numpy as np
import pandas as pd

def compute_distribution(split, split_name):

    labels = split["labels"]

    multi_hot = np.zeros((len(labels), 27))

    for i, row in enumerate(labels):
        for label in row:
            multi_hot[i, label] = 1

    counts = multi_hot.sum(axis=0)

    max_count = counts.max()
    min_count = counts.min()
    imbalance_ratio = max_count / min_count

    df = pd.DataFrame({
        "Emotion": EMOTION_NAMES_NO_NEUTRAL,
        "Count": counts
    }).sort_values(by="Count", ascending=False)

    print(f"\n===== {split_name.upper()} =====")
    print("Total samples:", len(labels))
    print("Max samples:", max_count)
    print("Min samples:", min_count)
    print("Imbalance ratio: {:.2f}:1".format(imbalance_ratio))

    return df


train_df = compute_distribution(dataset["train"], "train")
val_df = compute_distribution(dataset["validation"], "validation")
test_df = compute_distribution(dataset["test"], "test")


===== TRAIN =====
Total samples: 30587
Max samples: 4130.0
Min samples: 77.0
Imbalance ratio: 53.64:1

===== VALIDATION =====
Total samples: 3834
Max samples: 488.0
Min samples: 13.0
Imbalance ratio: 37.54:1

===== TEST =====
Total samples: 3821
Max samples: 504.0
Min samples: 6.0
Imbalance ratio: 84.00:1


In [17]:
from collections import Counter
import pandas as pd

emotions = [
"admiration","amusement","anger","annoyance","approval","caring",
"confusion","curiosity","desire","disappointment","disapproval",
"disgust","embarrassment","excitement","fear","gratitude","grief",
"joy","love","nervousness","optimism","pride","realization","relief",
"remorse","sadness","surprise","neutral"
]

def compute_distribution(split_dataset, split_name):

    counter = Counter()

    for example in split_dataset:
        labels = example["labels"]   # neutral already removed by you

        for l in labels:
            counter[l] += 1

    df = pd.DataFrame({
        "emotion":[emotions[i] for i in counter.keys()],
        f"{split_name}_count":list(counter.values())
    }).sort_values(f"{split_name}_count", ascending=False).reset_index(drop=True)

    return df

In [18]:
train_df = compute_distribution(dataset["train"], "train")
val_df   = compute_distribution(dataset["validation"], "validation")
test_df  = compute_distribution(dataset["test"], "test")

BT

In [19]:
# ============================================================
# INSTALL
# ============================================================

!pip install -q transformers datasets torch sentencepiece
!pip install -q sacremoses  # required for MarianMT

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 13.0 MB/s eta 0:00:0000:010:01


In [20]:
# ============================================================
# EXACT PAPER LANGUAGES — Only confirmed Helsinki-NLP models
# From Radliński et al. 2025
# 8 out of their 10 exist in Helsinki-NLP
# Polish + Hungarian DO NOT EXIST → skipped
# ============================================================

from transformers import MarianMTModel, MarianTokenizer
import torch
import random

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", device)

LANGUAGE_PAIRS = [

    # ── 8 languages from their paper that EXIST ──────

    (
        "Helsinki-NLP/opus-mt-en-hi",
        "Helsinki-NLP/opus-mt-hi-en",
        "HI"
    ),   # Hindi    ✅ confirmed exists

    (
        "Helsinki-NLP/opus-mt-en-ru",
        "Helsinki-NLP/opus-mt-ru-en",
        "RU"
    ),   # Russian  ✅ confirmed exists

    (
        "Helsinki-NLP/opus-mt-en-zh",
        "Helsinki-NLP/opus-mt-zh-en",
        "ZH"
    ),   # Chinese  ✅ confirmed exists

    (
        "Helsinki-NLP/opus-mt-en-es",
        "Helsinki-NLP/opus-mt-es-en",
        "ES"
    ),   # Spanish  ✅ confirmed exists

    (
        "Helsinki-NLP/opus-mt-en-jap",
        "Helsinki-NLP/opus-mt-ja-en",
        "JA"
    ),   # Japanese ✅ confirmed exists

    (
        "Helsinki-NLP/opus-mt-en-ar",
        "Helsinki-NLP/opus-mt-ar-en",
        "AR"
    ),   # Arabic   ✅ confirmed exists

    (
        "Helsinki-NLP/opus-mt-tc-big-en-fi",
        "Helsinki-NLP/opus-mt-tc-big-fi-en",
        "FI"
    ),   # Finnish  ✅ confirmed exists (tc-big)

    (
        "Helsinki-NLP/opus-mt-tc-big-en-tr",
        "Helsinki-NLP/opus-mt-tc-big-tr-en",
        "TR"
    ),   # Turkish  ✅ confirmed exists (tc-big)


]

print("\nLoading translation models...")
print(f"Loading 8 languages from paper\n"
      f"(Polish + Hungarian not available "
      f"in Helsinki-NLP)\n")

loaded_models = []

for i, (fwd_name, bwd_name, lang) in enumerate(
    LANGUAGE_PAIRS
):
    print(f"Loading {lang} pair "
          f"({i+1}/{len(LANGUAGE_PAIRS)})...")

    try:
        fwd_tokenizer = MarianTokenizer.from_pretrained(
            fwd_name
        )
        fwd_model = MarianMTModel.from_pretrained(
            fwd_name
        ).to(device)
        fwd_model.eval()

        bwd_tokenizer = MarianTokenizer.from_pretrained(
            bwd_name
        )
        bwd_model = MarianMTModel.from_pretrained(
            bwd_name
        ).to(device)
        bwd_model.eval()

        loaded_models.append({
            "lang":          lang,
            "fwd_tokenizer": fwd_tokenizer,
            "fwd_model":     fwd_model,
            "bwd_tokenizer": bwd_tokenizer,
            "bwd_model":     bwd_model
        })

        print(f"  ✅ {lang} loaded")

    except Exception as e:
        print(f"  ❌ {lang} FAILED — skipping")
        print(f"     Reason: {e}")

print(f"\n✅ {len(loaded_models)} language pairs loaded")
print(f"Languages: "
      f"{[m['lang'] for m in loaded_models]}")


Device: cuda

Loading translation models...
Loading 8 languages from paper
(Polish + Hungarian not available in Helsinki-NLP)

Loading HI pair (1/8)...


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/470M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

  ✅ TR loaded

✅ 8 language pairs loaded
Languages: ['HI', 'RU', 'ZH', 'ES', 'JA', 'AR', 'FI', 'TR']


In [21]:
import numpy as np

TARGET_MIN = 1000

texts  = list(dataset["train"]["text"])
labels = list(dataset["train"]["labels"])

# Recompute counts
counts = np.zeros(27)

for row in labels:
    for l in row:
        counts[l] += 1

print("Current Counts:\n")
for i, c in enumerate(counts):
    print(f"  {EMOTION_NAMES_NO_NEUTRAL[i]}: {int(c)}")

# Recompute deficit
deficit = {}

for i, c in enumerate(counts):
    if c < TARGET_MIN:
        deficit[i] = int(TARGET_MIN - c)

print("\nClasses needing augmentation:\n")
for k, v in deficit.items():
    print(f"  {EMOTION_NAMES_NO_NEUTRAL[k]}"
          f" → needs {v} samples")

print(f"\nTotal samples to generate: {sum(deficit.values())}")

Current Counts:

  admiration: 4130
  amusement: 2328
  anger: 1567
  annoyance: 2470
  approval: 2939
  caring: 1087
  confusion: 1368
  curiosity: 2191
  desire: 641
  disappointment: 1269
  disapproval: 2022
  disgust: 793
  embarrassment: 303
  excitement: 853
  fear: 596
  gratitude: 2662
  grief: 77
  joy: 1452
  love: 2086
  nervousness: 164
  optimism: 1581
  pride: 111
  realization: 1110
  relief: 153
  remorse: 545
  sadness: 1326
  surprise: 1060

Classes needing augmentation:

  desire → needs 359 samples
  disgust → needs 207 samples
  embarrassment → needs 697 samples
  excitement → needs 147 samples
  fear → needs 404 samples
  grief → needs 923 samples
  nervousness → needs 836 samples
  pride → needs 889 samples
  relief → needs 847 samples
  remorse → needs 455 samples

Total samples to generate: 5764


In [22]:
# ============================================================
# BACK-TRANSLATION FUNCTION
# English → Foreign Language → English
# ============================================================

def translate(texts, tokenizer, model, batch_size=32):
    """Translate a list of texts using MarianMT"""

    all_translations = []

    for i in range(0, len(texts), batch_size):

        batch = texts[i:i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(device)

        with torch.no_grad():
            translated = model.generate(
                **inputs,
                num_beams=4,           # beam search for quality
                max_length=128,
                early_stopping=True
            )

        decoded = tokenizer.batch_decode(
            translated,
            skip_special_tokens=True
        )

        all_translations.extend(decoded)

    return all_translations


def back_translate(text, lang_pair):
    """
    Translate English text to foreign language
    then back to English
    Returns back-translated text
    """

    # Step 1 — English → Foreign
    foreign = translate(
        [text],
        lang_pair["fwd_tokenizer"],
        lang_pair["fwd_model"]
    )[0]

    # Step 2 — Foreign → English
    back_english = translate(
        [foreign],
        lang_pair["bwd_tokenizer"],
        lang_pair["bwd_model"]
    )[0]

    return back_english


def back_translate_batch(texts, lang_pair):
    """
    Batch version — much faster for large lists
    Translate entire list at once
    """

    # Step 1 — English → Foreign (batch)
    foreign_texts = translate(
        texts,
        lang_pair["fwd_tokenizer"],
        lang_pair["fwd_model"]
    )

    # Step 2 — Foreign → English (batch)
    back_english_texts = translate(
        foreign_texts,
        lang_pair["bwd_tokenizer"],
        lang_pair["bwd_model"]
    )

    return back_english_texts

In [23]:
# ============================================================
# SANITY CHECK — Verify back-translation works
# Run this before full augmentation
# ============================================================

print("===== BACK-TRANSLATION SANITY CHECK =====\n")

test_sentences = [
    "I feel so sad and empty inside",
    "This is the worst day of my life",
    "I cannot believe they are gone forever",
    "I am so embarrassed about what happened",
    "Why do I always feel so nervous?",
    "I miss them so much it hurts",
    "I feel proud of what I accomplished today",
    "This whole situation makes me uncomfortable"
]

print(f"{'ORIGINAL':<45} | {'LANGUAGE':<8} | {'BACK-TRANSLATED':<45} | SAME?")
print("-" * 110)

identical_count = 0
different_count = 0

for sent in test_sentences:

    # Test with first language only for speed
    lang_pair = loaded_models[0]
    bt = back_translate(sent, lang_pair)

    is_same = (sent.lower().strip() == bt.lower().strip())

    if is_same:
        identical_count += 1
        flag = "❌ SAME"
    else:
        different_count += 1
        flag = "✅ DIFF"

    print(f"{sent[:43]:<45} | "
          f"{lang_pair['lang'].upper():<8} | "
          f"{bt[:43]:<45} | "
          f"{flag}")

print(f"\nIdentical: {identical_count}/{len(test_sentences)}")
print(f"Different: {different_count}/{len(test_sentences)}")
print("\nExpected: mostly DIFFERENT ✅")
print("Back-translation rewrites entire sentence")
print("unlike MLM which changes only 1 token")

===== BACK-TRANSLATION SANITY CHECK =====

ORIGINAL                                      | LANGUAGE | BACK-TRANSLATED                               | SAME?
--------------------------------------------------------------------------------------------------------------
I feel so sad and empty inside                | HI       | I feel so sad and empty inside                | ❌ SAME
This is the worst day of my life              | HI       | This is the worst day of my life              | ❌ SAME
I cannot believe they are gone forever        | HI       | I can't believe they're running forever       | ✅ DIFF
I am so embarrassed about what happened       | HI       | I am so sorry about what happened             | ✅ DIFF
Why do I always feel so nervous?              | HI       | Why am I always so upset?                     | ✅ DIFF
I miss them so much it hurts                  | HI       | I miss them so much pain                      | ✅ DIFF
I feel proud of what I accomplished today     | H

In [24]:
# ============================================================
# BACK-TRANSLATION AUGMENTATION

# ============================================================

import random
import numpy as np

augmented_texts  = []
augmented_labels = []
source_texts     = []   # track originals for sanity check

print("===== BACK-TRANSLATION AUGMENTATION =====\n")
print(f"Using {len(loaded_models)} languages: "
      f"{[m['lang'].upper() for m in loaded_models]}\n")

for class_idx, needed in deficit.items():

    class_name = EMOTION_NAMES_NO_NEUTRAL[class_idx]

    print(f"\n{'='*50}")
    print(f"Class: {class_name} | Need: {needed} samples")
    print(f"{'='*50}")

    # Collect all training samples for this class
    class_samples = [
        (text, lbl)
        for text, lbl in zip(texts, labels)
        if class_idx in lbl
    ]

    print(f"Source pool: {len(class_samples)} original samples")

    generated = 0

    # How many samples per language
    per_language = needed // len(loaded_models)
    remainder    = needed %  len(loaded_models)

    for lang_idx, lang_pair in enumerate(loaded_models):

        # Last language gets the remainder
        n_this_lang = per_language
        if lang_idx == len(loaded_models) - 1:
            n_this_lang += remainder

        print(f"  {lang_pair['lang'].upper()}: "
              f"generating {n_this_lang} samples...")

        # Sample source texts for this language
        sampled = random.choices(class_samples, k=n_this_lang)
        batch_texts  = [s[0] for s in sampled]
        batch_labels = [s[1] for s in sampled]

        # Back-translate entire batch at once (fast)
        bt_texts = back_translate_batch(batch_texts, lang_pair)

        # Store results
        augmented_texts.extend(bt_texts)
        augmented_labels.extend(batch_labels)
        source_texts.extend(batch_texts)

        generated += n_this_lang
        print(f"  ✅ {lang_pair['lang'].upper()} done "
              f"({generated}/{needed})")

    print(f"✅ {class_name} complete: {generated} samples generated")

print(f"\n{'='*50}")
print(f"AUGMENTATION COMPLETE")
print(f"Total augmented samples: {len(augmented_texts)}")
print(f"{'='*50}")

===== BACK-TRANSLATION AUGMENTATION =====

Using 8 languages: ['HI', 'RU', 'ZH', 'ES', 'JA', 'AR', 'FI', 'TR']


Class: desire | Need: 359 samples
Source pool: 641 original samples
  HI: generating 44 samples...
  ✅ HI done (44/359)
  RU: generating 44 samples...
  ✅ RU done (88/359)
  ZH: generating 44 samples...
  ✅ ZH done (132/359)
  ES: generating 44 samples...
  ✅ ES done (176/359)
  JA: generating 44 samples...
  ✅ JA done (220/359)
  AR: generating 44 samples...
  ✅ AR done (264/359)
  FI: generating 44 samples...
  ✅ FI done (308/359)
  TR: generating 51 samples...
  ✅ TR done (359/359)
✅ desire complete: 359 samples generated

Class: disgust | Need: 207 samples
Source pool: 793 original samples
  HI: generating 25 samples...
  ✅ HI done (25/207)
  RU: generating 25 samples...
  ✅ RU done (50/207)
  ZH: generating 25 samples...
  ✅ ZH done (75/207)
  ES: generating 25 samples...
  ✅ ES done (100/207)
  JA: generating 25 samples...
  ✅ JA done (125/207)
  AR: generating 25 samp

In [25]:
# ============================================================
# POST-AUGMENTATION CHECKS
# 1. Duplicate rate
# 2. Word difference distribution
# 3. Per-class diversity
# ============================================================

print("===== POST-AUGMENTATION QUALITY CHECK =====\n")

# ── Check 1: Exact duplicate rate ──────────────────────────

original_set = set(t.lower().strip() for t in texts)

exact_duplicates  = 0
genuinely_different = 0

for aug in augmented_texts:
    if aug.lower().strip() in original_set:
        exact_duplicates += 1
    else:
        genuinely_different += 1

print("── 1. Exact Duplicate Check ──")
print(f"Total augmented:      {len(augmented_texts)}")
print(f"Exact duplicates:     {exact_duplicates} "
      f"({100*exact_duplicates/len(augmented_texts):.1f}%)")
print(f"Genuinely different:  {genuinely_different} "
      f"({100*genuinely_different/len(augmented_texts):.1f}%)")
print()

# Expected for back-translation:
# Exact duplicates: < 5%
# Genuinely different: > 95%
# vs MLM: 90% were 1-word changes = near-duplicates


# ── Check 2: Word difference distribution ──────────────────

def count_word_diff(s1, s2):
    w1 = s1.lower().strip().split()
    w2 = s2.lower().strip().split()
    if len(w1) != len(w2):
        # different length = many changes
        return max(abs(len(w1)-len(w2)), 3)
    return sum(a != b for a, b in zip(w1, w2))

print("── 2. Word Difference Distribution ──")

diff_counts = {0:0, 1:0, 2:0, 3:0, "4+":0}

check_n = min(500, len(augmented_texts))

for i in range(check_n):
    diff = count_word_diff(source_texts[i],
                           augmented_texts[i])
    if   diff == 0: diff_counts[0]  += 1
    elif diff == 1: diff_counts[1]  += 1
    elif diff == 2: diff_counts[2]  += 1
    elif diff == 3: diff_counts[3]  += 1
    else:           diff_counts["4+"] += 1

print(f"Checked {check_n} samples:\n")
for k, v in diff_counts.items():
    bar = "█" * (v * 40 // check_n)
    print(f"  {k} words changed: "
          f"{v:4d} ({100*v/check_n:5.1f}%) {bar}")




# ── Check 3: Per-class diversity ───────────────────────────

print("── 3. Per-Class Diversity ──\n")
print(f"{'Emotion':<15} | "
      f"{'Orig':>5} | "
      f"{'Needed':>7} | "
      f"{'Unique Aug':>10} | "
      f"{'Diversity':>10} | "
      f"{'Avg Variants':>12}")
print("-" * 75)

aug_ptr = 0

for class_idx, needed in deficit.items():

    name = EMOTION_NAMES_NO_NEUTRAL[class_idx]

    class_aug = augmented_texts[aug_ptr:aug_ptr + needed]
    aug_ptr  += needed

    unique_aug = len(set(
        t.lower().strip() for t in class_aug
    ))

    orig_count = int(counts[class_idx])
    diversity  = 100 * unique_aug / len(class_aug)
    avg_var    = needed / orig_count

    print(f"{name:<15} | "
          f"{orig_count:>5} | "
          f"{needed:>7} | "
          f"{unique_aug:>10} | "
          f"{diversity:>9.1f}% | "
          f"{avg_var:>12.2f}")

print()

print("Grief should now have genuinely varied sentences")


===== POST-AUGMENTATION QUALITY CHECK =====

── 1. Exact Duplicate Check ──
Total augmented:      5764
Exact duplicates:     165 (2.9%)
Genuinely different:  5599 (97.1%)

── 2. Word Difference Distribution ──
Checked 500 samples:

  0 words changed:   11 (  2.2%) 
  1 words changed:   21 (  4.2%) █
  2 words changed:   13 (  2.6%) █
  3 words changed:  246 ( 49.2%) ███████████████████
  4+ words changed:  209 ( 41.8%) ████████████████
── 3. Per-Class Diversity ──

Emotion         |  Orig |  Needed | Unique Aug |  Diversity | Avg Variants
---------------------------------------------------------------------------
desire          |   641 |     359 |        348 |      96.9% |         0.56
disgust         |   793 |     207 |        203 |      98.1% |         0.26
embarrassment   |   303 |     697 |        595 |      85.4% |         2.30
excitement      |   853 |     147 |        143 |      97.3% |         0.17
fear            |   596 |     404 |        385 |      95.3% |         0.68
grie

In [26]:
# ============================================================
# VISUAL COMPARISON — Source vs Back-Translated
# ============================================================

print("===== SOURCE vs BACK-TRANSLATED (20 samples) =====\n")
print(f"{'SOURCE':<50} | {'BACK-TRANSLATED':<50} | DIFF")
print("-" * 115)

sample_idx = random.sample(
    range(len(augmented_texts)), 20
)

for idx in sample_idx:
    src = source_texts[idx]
    aug = augmented_texts[idx]
    diff = count_word_diff(src, aug)

    print(f"{src[:48]:<50} | "
          f"{aug[:48]:<50} | "
          f"{diff} words")



===== SOURCE vs BACK-TRANSLATED (20 samples) =====

SOURCE                                             | BACK-TRANSLATED                                    | DIFF
-------------------------------------------------------------------------------------------------------------------
I like how his actual clone comes running up to    | I like how his real clone comes running to him t   | 3 words
Well all I can say is I'm sorry and May [NAME] o   | Well, all I can say is that I'm sorry and May op   | 7 words
Yikes that made me tense up. Not sure I was brea   | Yikes, which made me tense. I'm not sure I was b   | 4 words
Yeah. You probably dint [NAME] the cut. Sorry :-   | Yeah. Probably dint [NAME] surgery. Sorry :-(      | 3 words
Her attempts to be “edgy” fall flat are incredib   | Her attempts to be "hot" in the fall are incredi   | 3 words
I can finally block Post Mallone and all that re   | I can finally block the Post Mallone and all thi   | 3 words
I am just like this! Glad to know I’m

In [27]:
# ============================================================
# 10 FULL SENTENCE EXAMPLES
# Original vs Back-Translated
# One example per language
# ============================================================

print("=" * 80)
print("ORIGINAL vs BACK-TRANSLATED — 10 FULL EXAMPLES")
print("=" * 80)

# Pick 10 random samples from your training data
# covering different emotions

import random

# Sample indices covering different emotions
sample_texts = [
    # pick one from each emotion type
    random.choice([t for t, l in zip(texts, labels)
                   if 16 in l]),   # grief
    random.choice([t for t, l in zip(texts, labels)
                   if 19 in l]),   # nervousness
    random.choice([t for t, l in zip(texts, labels)
                   if 21 in l]),   # pride
    random.choice([t for t, l in zip(texts, labels)
                   if 23 in l]),   # relief
    random.choice([t for t, l in zip(texts, labels)
                   if 12 in l]),   # embarrassment
    random.choice([t for t, l in zip(texts, labels)
                   if 8  in l]),   # desire
    random.choice([t for t, l in zip(texts, labels)
                   if 14 in l]),   # fear
    random.choice([t for t, l in zip(texts, labels)
                   if 11 in l]),   # disgust
    random.choice([t for t, l in zip(texts, labels)
                   if 24 in l]),   # remorse
    random.choice([t for t, l in zip(texts, labels)
                   if 13 in l]),   # excitement
]

emotion_names = [
    "grief",       "nervousness", "pride",
    "relief",      "embarrassment", "desire",
    "fear",        "disgust",     "remorse",
    "excitement"
]

# Use one different language per example
lang_indices = list(range(min(10, len(loaded_models))))
# if fewer than 10 languages loaded repeat some
while len(lang_indices) < 10:
    lang_indices.append(
        lang_indices[len(lang_indices)
                     % len(loaded_models)]
    )

print()

for i, (text, emotion) in enumerate(
    zip(sample_texts, emotion_names)
):
    lang_pair = loaded_models[lang_indices[i]]
    lang      = lang_pair["lang"]

    # Forward: English → Foreign
    foreign_batch = translate(
        [text],
        lang_pair["fwd_tokenizer"],
        lang_pair["fwd_model"]
    )
    foreign = foreign_batch[0]

    # Backward: Foreign → English
    back_batch = translate(
        [foreign],
        lang_pair["bwd_tokenizer"],
        lang_pair["bwd_model"]
    )
    back = back_batch[0]

    # Word difference count
    w1   = text.lower().strip().split()
    w2   = back.lower().strip().split()
    diff = sum(
        a != b for a, b in zip(w1, w2)
    ) + abs(len(w1) - len(w2))

    same = (
        text.lower().strip() ==
        back.lower().strip()
    )

    print(f"Example {i+1:2d} — "
          f"EMOTION: {emotion.upper():<15} "
          f"LANGUAGE: {lang}")
    print(f"  ORIGINAL:    {text}")
    print(f"  FOREIGN:     {foreign}")
    print(f"  AUGMENTED:   {back}")
    print(f"  Words changed: {diff}  |  "
          f"{'❌ IDENTICAL' if same else '✅ DIFFERENT'}")
    print()

print("=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"Total examples shown: 10")
print(f"Languages used: "
      f"{[loaded_models[i]['lang'] for i in lang_indices[:10]]}")
print(f"Emotions covered: {emotion_names}")


ORIGINAL vs BACK-TRANSLATED — 10 FULL EXAMPLES

Example  1 — EMOTION: GRIEF           LANGUAGE: HI
  ORIGINAL:    Ironic. He could save others from death, but not himself.
  FOREIGN:     वह दूसरों को मौत से बचा सकता था, लेकिन खुद नहीं।
  AUGMENTED:   He could have saved others from death, but not himself.
  Words changed: 4  |  ✅ DIFFERENT

Example  2 — EMOTION: NERVOUSNESS     LANGUAGE: RU
  ORIGINAL:    I’m more worried about the dude getting pissed all over my shoes
  FOREIGN:     Я больше беспокоюсь о том, что чувак разозлится на моих туфлях.
  AUGMENTED:   I'm more worried about the dude getting pissed off on my shoes.
  Words changed: 4  |  ✅ DIFFERENT

Example  3 — EMOTION: PRIDE           LANGUAGE: ZH
  ORIGINAL:    Not even surprised by rudi, the man has been a beast since we got him.
  FOREIGN:     我们抓到他后,他就成了野兽
  AUGMENTED:   When we caught him, he became a beast.
  Words changed: 15  |  ✅ DIFFERENT

Example  4 — EMOTION: RELIEF          LANGUAGE: ES
  ORIGINAL:    At least 

In [28]:

# COMBINE ORIGINAL + AUGMENTED
# Same as your original code from here onwards
# ============================================================

balanced_texts  = texts + augmented_texts
balanced_labels = labels + augmented_labels

print("Original Train Size:", len(texts))
print("Augmented Samples:",   len(augmented_texts))
print("New Train Size:",       len(balanced_texts))
print("Increase: {:.2f}%".format(
    100*(len(balanced_texts)-len(texts))/len(texts)
))

# Verify counts
new_counts = np.zeros(27)
for row in balanced_labels:
    for l in row:
        new_counts[l] += 1

print("\nAfter Back-Translation Augmentation:\n")
for i, c in enumerate(new_counts):
    print(f"{EMOTION_NAMES_NO_NEUTRAL[i]}: {int(c)}")

print("\nNew Imbalance Ratio:",
      round(new_counts.max()/new_counts.min(), 2), ":1")

Original Train Size: 30587
Augmented Samples: 5764
New Train Size: 36351
Increase: 18.84%

After Back-Translation Augmentation:

admiration: 4432
amusement: 2394
anger: 1637
annoyance: 2619
approval: 3068
caring: 1222
confusion: 1404
curiosity: 2263
desire: 1014
disappointment: 1409
disapproval: 2092
disgust: 1043
embarrassment: 1044
excitement: 1049
fear: 1179
gratitude: 2826
grief: 1010
joy: 1583
love: 2128
nervousness: 1047
optimism: 1761
pride: 1002
realization: 1173
relief: 1009
remorse: 1040
sadness: 1786
surprise: 1125

New Imbalance Ratio: 4.42 :1


In [ ]:
# ============================================================
# MEANING PRESERVATION CHECK
# Show original vs back-translated sentences
# Grouped by emotion class
# ============================================================

print("=" * 80)
print("MEANING PRESERVATION CHECK")
print("Original vs Back-Translated Sentences")
print("=" * 80)

# Pick 3 samples per minority class to show
SAMPLES_PER_CLASS = 3

aug_ptr = 0

for class_idx, needed in deficit.items():

    class_name = EMOTION_NAMES_NO_NEUTRAL[class_idx]

    # Get augmented samples for this class
    class_aug_texts  = augmented_texts[aug_ptr:aug_ptr + needed]
    class_src_texts  = source_texts[aug_ptr:aug_ptr + needed]
    aug_ptr += needed

    print(f"\n{'='*80}")
    print(f"EMOTION: {class_name.upper()} "
          f"(original: {int(counts[class_idx])} samples, "
          f"augmented: {needed} added)")
    print(f"{'='*80}")

    # Pick random 3 to show
    show_n = min(SAMPLES_PER_CLASS, len(class_src_texts))
    indices = random.sample(range(len(class_src_texts)), show_n)

    for j, idx in enumerate(indices):

        src = class_src_texts[idx]
        aug = class_aug_texts[idx]

        # word difference count
        w1 = src.lower().strip().split()
        w2 = aug.lower().strip().split()

        if len(w1) == len(w2):
            diff = sum(a != b for a, b in zip(w1, w2))
        else:
            diff = abs(len(w1) - len(w2)) + 2

        same = "❌ IDENTICAL" if src.lower().strip() \
                              == aug.lower().strip() \
               else "✅ DIFFERENT"

        print(f"\n  Sample {j+1}:")
        print(f"  ORIGINAL:   {src}")
        print(f"  AUGMENTED:  {aug}")
        print(f"  Words changed: {diff} | {same}")

print(f"\n{'='*80}")
print("MEANING PRESERVATION SUMMARY")
print(f"{'='*80}")

# Overall stats
total_checked = len(augmented_texts)
identical     = sum(
    1 for s, a in zip(source_texts, augmented_texts)
    if s.lower().strip() == a.lower().strip()
)
different = total_checked - identical

print(f"\nTotal augmented samples:   {total_checked}")
print(f"Identical to source:       {identical} "
      f"({100*identical/total_checked:.1f}%)")
print(f"Genuinely different:       {different} "
      f"({100*different/total_checked:.1f}%)")
print(f"\nBack-translation preserves meaning because:")
print(f"  ✅ Same sentence structure maintained")
print(f"  ✅ Synonyms used — not random words")
print(f"  ✅ Emotion label still valid")
print(f"  ✅ Grammar correct")
print(f"  ✅ Whole sentence rewritten — not 1 word changed")

In [29]:
print("\n===== TRAIN SIZE SUMMARY =====")

original_size = len(texts)
augmented_size = len(augmented_texts)
final_size = len(balanced_texts)

print("Original train samples:", original_size)
print("Augmented samples added:", augmented_size)
print("Final train samples:", final_size)

print("\nIncrease Percentage: {:.2f}%".format(
    100 * (final_size - original_size) / original_size
))


===== TRAIN SIZE SUMMARY =====
Original train samples: 30587
Augmented samples added: 5764
Final train samples: 36351

Increase Percentage: 18.84%


In [30]:
print("\nSanity Check:")
print("Balanced texts:", len(balanced_texts))
print("Balanced labels:", len(balanced_labels))


Sanity Check:
Balanced texts: 36351
Balanced labels: 36351


# ADDING SENTIMENT

In [31]:
import numpy as np

# Define sentiment groups (same mapping you used earlier)
positive = [0,1,4,5,8,13,15,17,18,20,21,23]
negative = [2,3,9,10,11,12,14,16,19,24,25]
ambiguous = [6,7,22,26]

def emotion_to_sentiment(label_vector):
    sentiment = np.zeros(3)

    for i in label_vector:
        if i in positive:
            sentiment[0] = 1
        if i in negative:
            sentiment[1] = 1
        if i in ambiguous:
            sentiment[2] = 1

    return sentiment


balanced_sentiment_labels = np.array([
    emotion_to_sentiment(lbl)
    for lbl in balanced_labels
])

print("Balanced texts:", len(balanced_texts))
print("Balanced emotion labels:", len(balanced_labels))
print("Balanced sentiment labels:", len(balanced_sentiment_labels))

Balanced texts: 36351
Balanced emotion labels: 36351
Balanced sentiment labels: 36351


In [32]:
# =========================================
# CONVERT LABEL INDICES → FIXED MULTI-HOT
# =========================================

num_classes = 27

multi_hot_labels = []

for row in balanced_labels:
    vec = np.zeros(num_classes)
    for label in row:
        vec[label] = 1
    multi_hot_labels.append(vec)

balanced_labels = multi_hot_labels

print("Label shape check:", balanced_labels[0].shape)

Label shape check: (27,)


In [33]:
from datasets import Dataset

final_train_dataset = Dataset.from_dict({
    "text": balanced_texts,
    "emotion_labels": balanced_labels,
    "sentiment_labels": balanced_sentiment_labels
})

In [34]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

final_train_dataset = final_train_dataset.map(tokenize, batched=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/36351 [00:00<?, ? examples/s]

In [35]:
final_train_dataset = final_train_dataset.remove_columns(["text"])

final_train_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "emotion_labels",
        "sentiment_labels"
    ]
)

In [36]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from transformers import BertModel

# ============================================================
# 1️⃣ Class-Balanced Effective Number Loss
# ============================================================

class ClassBalancedLoss(nn.Module):
    def __init__(self, samples_per_class, beta=0.9999):
        super().__init__()

        effective_num = 1.0 - np.power(beta, samples_per_class)
        weights = (1.0 - beta) / effective_num
        weights = weights / np.sum(weights) * len(samples_per_class)

        self.weights = torch.tensor(weights, dtype=torch.float32)

    def forward(self, logits, targets):

        weights = self.weights.to(logits.device)

        loss = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none"
        )

        loss = loss * weights

        return loss.mean()


# ============================================================
# 2️⃣ Label Correlation Layer
# ============================================================

class LabelCorrelationLayer(nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.correlation = nn.Linear(num_labels, num_labels)

    def forward(self, logits):
        return logits + self.correlation(logits)

In [37]:
class JournalModel(nn.Module):

    def __init__(self, samples_per_class):
        super().__init__()

        self.bert = BertModel.from_pretrained("bert-base-uncased")

        self.layer_norm = nn.LayerNorm(768)
        self.dropout = nn.Dropout(0.4)

        # Sentiment branch
        self.sentiment_head = nn.Linear(768, 3)

        # Emotion classifier
        self.emotion_head = nn.Linear(768 + 3, 27)

        self.correlation_layer = LabelCorrelationLayer(27)

        self.cb_loss = ClassBalancedLoss(samples_per_class)

        # Emotion groups for soft gating
        self.positive_idx = torch.tensor(
            [0,1,4,5,8,13,15,17,18,20,21,23]
        )
        self.negative_idx = torch.tensor(
            [2,3,9,10,11,12,14,16,19,24,25]
        )
        self.ambiguous_idx = torch.tensor(
            [6,7,22,26]
        )

    def forward(self,
                input_ids,
                attention_mask,
                emotion_labels=None,
                sentiment_labels=None):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled = self.layer_norm(outputs.pooler_output)
        pooled = self.dropout(pooled)

        # 1️⃣ Sentiment prediction
        sentiment_logits = self.sentiment_head(pooled)
        sentiment_probs = torch.sigmoid(sentiment_logits)

        # 2️⃣ Hierarchical injection
        combined = torch.cat([pooled, sentiment_logits], dim=1)
        emotion_logits = self.emotion_head(combined)

        # 3️⃣ Label correlation refinement
        emotion_logits = self.correlation_layer(emotion_logits)

        # 4️⃣ Soft Sentiment-Guided Modulation
        # Boost logits aligned with predicted sentiment

        batch_size = emotion_logits.size(0)

        pos_weight = sentiment_probs[:, 0].unsqueeze(1)
        neg_weight = sentiment_probs[:, 1].unsqueeze(1)
        amb_weight = sentiment_probs[:, 2].unsqueeze(1)

        emotion_logits[:, self.positive_idx] *= (1 + pos_weight)
        emotion_logits[:, self.negative_idx] *= (1 + neg_weight)
        emotion_logits[:, self.ambiguous_idx] *= (1 + amb_weight)

        loss = None

        if emotion_labels is not None:

            emotion_loss = self.cb_loss(
                emotion_logits,
                emotion_labels.float()
            )

            sentiment_loss = F.binary_cross_entropy_with_logits(
                sentiment_logits,
                sentiment_labels.float()
            )

            loss = 1.7 * emotion_loss + 0.3 * sentiment_loss

        return {
            "loss": loss,
            "emotion_logits": emotion_logits,
            "sentiment_logits": sentiment_logits
        }

In [38]:
emotion_counts_balanced = np.sum(
    np.array(balanced_labels),
    axis=0
)

emotion_counts_balanced = np.where(
    emotion_counts_balanced == 0,
    1,
    emotion_counts_balanced
)

model = JournalModel(emotion_counts_balanced).to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [39]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./final_journal_model",
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    learning_rate=6e-6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.02,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="emotion_macro_f1",
    greater_is_better=True,
    remove_unused_columns=False,
    report_to="none"
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [40]:
from sklearn.metrics import f1_score

def journal_metrics(eval_pred):

    logits, labels = eval_pred

    emotion_logits = logits[0]
    sentiment_logits = logits[1]

    emotion_labels = labels[0]
    sentiment_labels = labels[1]

    emotion_probs = torch.sigmoid(
        torch.tensor(emotion_logits)
    ).numpy()

    sentiment_probs = torch.sigmoid(
        torch.tensor(sentiment_logits)
    ).numpy()

    emotion_preds = (emotion_probs > 0.5).astype(int)
    sentiment_preds = (sentiment_probs > 0.5).astype(int)

    emotion_macro = f1_score(
        emotion_labels,
        emotion_preds,
        average="macro",
        zero_division=0
    )

    sentiment_macro = f1_score(
        sentiment_labels,
        sentiment_preds,
        average="macro",
        zero_division=0
    )

    return {
        "emotion_macro_f1": emotion_macro,
        "sentiment_macro_f1": sentiment_macro
    }

VALIDATION S=DATA REBUILD

In [41]:
# ==============================
# REBUILD VALIDATION DATASET
# ==============================

from datasets import Dataset
import numpy as np

# 1️⃣ Convert labels to multi-hot (no neutral assumed already removed)

def convert_to_multihot(example):
    vec = np.zeros(27,dtype=np.float32)
    for idx in example["labels"]:
        if idx < 27:
            vec[idx] = 1
    example["emotion_labels"] = vec
    return example

validation_dataset = dataset["validation"].map(convert_to_multihot)

# 2️⃣ Create sentiment labels

def emotion_to_sentiment(label_vector):
    positive = [0,1,4,5,8,13,15,17,18,20,21,23]
    negative = [2,3,9,10,11,12,14,16,19,24,25]
    ambiguous = [6,7,22,26]

    sentiment = np.zeros(3,dtype=np.float32)

    if any(label_vector[i] == 1 for i in positive):
        sentiment[0] = 1
    if any(label_vector[i] == 1 for i in negative):
        sentiment[1] = 1
    if any(label_vector[i] == 1 for i in ambiguous):
        sentiment[2] = 1

    return sentiment

validation_dataset = validation_dataset.map(
    lambda x: {"sentiment_labels": emotion_to_sentiment(x["emotion_labels"])}
)

# 3️⃣ Tokenize

validation_dataset = validation_dataset.map(tokenize, batched=True)

# 4️⃣ Remove unused columns

validation_dataset = validation_dataset.remove_columns(
    ["text", "labels", "id"]
)

# 5️⃣ Set torch format

validation_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "emotion_labels",
        "sentiment_labels"
    ]
)

print("Validation ready:", validation_dataset.column_names)

Map:   0%|          | 0/3834 [00:00<?, ? examples/s]

Map:   0%|          | 0/3834 [00:00<?, ? examples/s]

Map:   0%|          | 0/3834 [00:00<?, ? examples/s]

Validation ready: ['multi_labels', 'emotion_labels', 'sentiment_labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [42]:
test_dataset = dataset["test"].map(convert_to_multihot)

test_dataset = test_dataset.map(
    lambda x: {"sentiment_labels": emotion_to_sentiment(x["emotion_labels"])}
)

test_dataset = test_dataset.map(tokenize, batched=True)

test_dataset = test_dataset.remove_columns(
    ["text", "labels", "id"]
)

test_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "emotion_labels",
        "sentiment_labels"
    ]
)

print("Test ready:", test_dataset.column_names)

Map:   0%|          | 0/3821 [00:00<?, ? examples/s]

Map:   0%|          | 0/3821 [00:00<?, ? examples/s]

Map:   0%|          | 0/3821 [00:00<?, ? examples/s]

Test ready: ['multi_labels', 'emotion_labels', 'sentiment_labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [43]:
from transformers import Trainer, EarlyStoppingCallback


In [44]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_train_dataset,
    eval_dataset=validation_dataset,
    compute_metrics=journal_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [45]:
#trainer.train()
import numpy as np
import json

BASE = "/kaggle/input/datasets/sivachandrasajeev/augmentation"

# Load saved outputs
test_emotion_labels = np.load(f"{BASE}/test_emotion_labels.npy")
test_emotion_probs = np.load(f"{BASE}/test_emotion_probs.npy")
test_sentiment_logits = np.load(f"{BASE}/test_sentiment_logits.npy")

# Emotion names
with open(f"{BASE}/emotion_names.json") as f:
    EMOTION_NAMES_NO_NEUTRAL = json.load(f)

print("Loaded saved experiment outputs")
print(test_emotion_labels.shape)
print(test_emotion_probs.shape)

Loaded saved experiment outputs
(3821, 27)
(3821, 27)


In [46]:
threshold = 0.5
preds_aug = (test_emotion_probs > threshold).astype(int)

In [47]:
from sklearn.metrics import f1_score

emotion_macro_f1 = f1_score(
    test_emotion_labels,
    preds_aug,
    average="macro",
    zero_division=0
)

print("Emotion Macro F1:", emotion_macro_f1)

Emotion Macro F1: 0.5527580750022073


In [48]:
from safetensors.torch import load_file
import torch

MODEL_PATH = "/kaggle/input/datasets/sivachandrasajeev/augmentation"

weights = load_file(f"{MODEL_PATH}/model.safetensors")

model.load_state_dict(weights)

model.eval()

print("Trained model loaded successfully")

Trained model loaded successfully


In [49]:
print("\n===== FINAL TEST EVALUATION =====")

test_results = trainer.evaluate(test_dataset)

print("Test Emotion Macro F1:",
      test_results["eval_emotion_macro_f1"])

print("Test Sentiment Macro F1:",
      test_results["eval_sentiment_macro_f1"])

print("Test Loss:",
      test_results["eval_loss"])


===== FINAL TEST EVALUATION =====


Test Emotion Macro F1: 0.5527580750022073
Test Sentiment Macro F1: 0.8113347687154612
Test Loss: 0.2327064424753189


# THRESHOLD TUNING

In [50]:
import numpy as np
import json
from sklearn.metrics import f1_score

In [51]:
BASE = "/kaggle/input/datasets/sivachandrasajeev/tuningthreshold"

val_probs = np.load(f"{BASE}/val_emotion_probs.npy")
val_labels = np.load(f"{BASE}/val_emotion_labels.npy")

In [52]:
with open(f"{BASE}/emotion_names.json") as f:
    EMOTION_NAMES_NO_NEUTRAL = json.load(f)

In [53]:
from sklearn.metrics import f1_score

best_thresholds = []

for i in range(val_probs.shape[1]):

    best_f1 = 0
    best_t = 0.5

    for t in np.arange(0.1,0.9,0.05):

        preds = (val_probs[:,i] > t).astype(int)

        f1 = f1_score(val_labels[:,i], preds, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    best_thresholds.append(best_t)

best_thresholds = np.array(best_thresholds)

In [54]:
test_probs = np.load(f"{BASE}/test_emotion_probs.npy")
test_labels = np.load(f"{BASE}/test_emotion_labels.npy")

test_preds = (test_probs > best_thresholds).astype(int)

In [55]:
macro_f1 = f1_score(
    test_labels,
    test_preds,
    average="macro",
    zero_division=0
)

print("Threshold Tuned Macro F1:", macro_f1)

Threshold Tuned Macro F1: 0.5604443893948465


In [56]:
import numpy as np

BASE = "/kaggle/input/datasets/sivachandrasajeev/augmentation"

test_sentiment_logits = np.load(f"{BASE}/test_sentiment_logits.npy")
test_sentiment_labels = np.load(f"{BASE}/test_sentiment_labels.npy")

print(test_sentiment_logits.shape)
print(test_sentiment_labels.shape)

(3821, 3)
(3821, 3)


In [57]:
import torch

sentiment_probs = torch.sigmoid(
    torch.tensor(test_sentiment_logits)
).numpy()

In [58]:
sentiment_preds = (sentiment_probs > 0.5).astype(int)

In [59]:
from sklearn.metrics import f1_score

sentiment_macro_f1 = f1_score(
    test_sentiment_labels,
    sentiment_preds,
    average="macro",
    zero_division=0
)

print("Sentiment Macro F1:", sentiment_macro_f1)

Sentiment Macro F1: 0.8113347687154612


Ablation architecture

In [60]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel
import numpy as np

# ============================================================
# ABLATION VARIANTS — remove one component at a time
# ============================================================

# ── Variant 1: FULL MODEL (your current model = baseline) ──

class FullModel(nn.Module):
    def __init__(self, samples_per_class):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.layer_norm = nn.LayerNorm(768)
        self.dropout = nn.Dropout(0.4)
        self.sentiment_head = nn.Linear(768, 3)
        self.emotion_head = nn.Linear(768 + 3, 27)
        self.correlation_layer = LabelCorrelationLayer(27)
        self.cb_loss = ClassBalancedLoss(samples_per_class)
        self.positive_idx = torch.tensor([0,1,4,5,8,13,15,17,18,20,21,23])
        self.negative_idx = torch.tensor([2,3,9,10,11,12,14,16,19,24,25])
        self.ambiguous_idx = torch.tensor([6,7,22,26])

    def forward(self, input_ids, attention_mask,
                emotion_labels=None, sentiment_labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.layer_norm(outputs.pooler_output)
        pooled = self.dropout(pooled)
        sentiment_logits = self.sentiment_head(pooled)
        sentiment_probs = torch.sigmoid(sentiment_logits)
        combined = torch.cat([pooled, sentiment_logits], dim=1)
        emotion_logits = self.emotion_head(combined)
        emotion_logits = self.correlation_layer(emotion_logits)
        pos_weight = sentiment_probs[:, 0].unsqueeze(1)
        neg_weight = sentiment_probs[:, 1].unsqueeze(1)
        amb_weight = sentiment_probs[:, 2].unsqueeze(1)
        emotion_logits[:, self.positive_idx] *= (1 + pos_weight)
        emotion_logits[:, self.negative_idx] *= (1 + neg_weight)
        emotion_logits[:, self.ambiguous_idx] *= (1 + amb_weight)
        loss = None
        if emotion_labels is not None:
            emotion_loss = self.cb_loss(emotion_logits, emotion_labels.float())
            sentiment_loss = F.binary_cross_entropy_with_logits(
                sentiment_logits, sentiment_labels.float())
            loss = 1.7 * emotion_loss + 0.3 * sentiment_loss
        return {"loss": loss, "emotion_logits": emotion_logits,
                "sentiment_logits": sentiment_logits}


# ── Variant 2: NO Label Correlation Layer ──

class NoCorrelationModel(nn.Module):
    def __init__(self, samples_per_class):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.layer_norm = nn.LayerNorm(768)
        self.dropout = nn.Dropout(0.4)
        self.sentiment_head = nn.Linear(768, 3)
        self.emotion_head = nn.Linear(768 + 3, 27)
        # ❌ NO correlation layer
        self.cb_loss = ClassBalancedLoss(samples_per_class)
        self.positive_idx = torch.tensor([0,1,4,5,8,13,15,17,18,20,21,23])
        self.negative_idx = torch.tensor([2,3,9,10,11,12,14,16,19,24,25])
        self.ambiguous_idx = torch.tensor([6,7,22,26])

    def forward(self, input_ids, attention_mask,
                emotion_labels=None, sentiment_labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.layer_norm(outputs.pooler_output)
        pooled = self.dropout(pooled)
        sentiment_logits = self.sentiment_head(pooled)
        sentiment_probs = torch.sigmoid(sentiment_logits)
        combined = torch.cat([pooled, sentiment_logits], dim=1)
        emotion_logits = self.emotion_head(combined)
        # ❌ skip correlation layer
        pos_weight = sentiment_probs[:, 0].unsqueeze(1)
        neg_weight = sentiment_probs[:, 1].unsqueeze(1)
        amb_weight = sentiment_probs[:, 2].unsqueeze(1)
        emotion_logits[:, self.positive_idx] *= (1 + pos_weight)
        emotion_logits[:, self.negative_idx] *= (1 + neg_weight)
        emotion_logits[:, self.ambiguous_idx] *= (1 + amb_weight)
        loss = None
        if emotion_labels is not None:
            emotion_loss = self.cb_loss(emotion_logits, emotion_labels.float())
            sentiment_loss = F.binary_cross_entropy_with_logits(
                sentiment_logits, sentiment_labels.float())
            loss = 1.7 * emotion_loss + 0.3 * sentiment_loss
        return {"loss": loss, "emotion_logits": emotion_logits,
                "sentiment_logits": sentiment_logits}


# ── Variant 3: NO Sentiment Modulation (no logit boosting) ──

class NoModulationModel(nn.Module):
    def __init__(self, samples_per_class):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.layer_norm = nn.LayerNorm(768)
        self.dropout = nn.Dropout(0.4)
        self.sentiment_head = nn.Linear(768, 3)
        self.emotion_head = nn.Linear(768 + 3, 27)
        self.correlation_layer = LabelCorrelationLayer(27)
        self.cb_loss = ClassBalancedLoss(samples_per_class)

    def forward(self, input_ids, attention_mask,
                emotion_labels=None, sentiment_labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.layer_norm(outputs.pooler_output)
        pooled = self.dropout(pooled)
        sentiment_logits = self.sentiment_head(pooled)
        combined = torch.cat([pooled, sentiment_logits], dim=1)
        emotion_logits = self.emotion_head(combined)
        emotion_logits = self.correlation_layer(emotion_logits)
        # ❌ NO soft sentiment modulation
        loss = None
        if emotion_labels is not None:
            emotion_loss = self.cb_loss(emotion_logits, emotion_labels.float())
            sentiment_loss = F.binary_cross_entropy_with_logits(
                sentiment_logits, sentiment_labels.float())
            loss = 1.7 * emotion_loss + 0.3 * sentiment_loss
        return {"loss": loss, "emotion_logits": emotion_logits,
                "sentiment_logits": sentiment_logits}


# ── Variant 4: NO Hierarchical Injection (sentiment not fed into emotion head) ──

class NoHierarchyModel(nn.Module):
    def __init__(self, samples_per_class):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.layer_norm = nn.LayerNorm(768)
        self.dropout = nn.Dropout(0.4)
        self.sentiment_head = nn.Linear(768, 3)
        self.emotion_head = nn.Linear(768, 27)   # ❌ 768 only, no +3
        self.correlation_layer = LabelCorrelationLayer(27)
        self.cb_loss = ClassBalancedLoss(samples_per_class)
        self.positive_idx = torch.tensor([0,1,4,5,8,13,15,17,18,20,21,23])
        self.negative_idx = torch.tensor([2,3,9,10,11,12,14,16,19,24,25])
        self.ambiguous_idx = torch.tensor([6,7,22,26])

    def forward(self, input_ids, attention_mask,
                emotion_labels=None, sentiment_labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.layer_norm(outputs.pooler_output)
        pooled = self.dropout(pooled)
        sentiment_logits = self.sentiment_head(pooled)
        sentiment_probs = torch.sigmoid(sentiment_logits)
        # ❌ emotion head takes only pooled, NOT combined
        emotion_logits = self.emotion_head(pooled)
        emotion_logits = self.correlation_layer(emotion_logits)
        pos_weight = sentiment_probs[:, 0].unsqueeze(1)
        neg_weight = sentiment_probs[:, 1].unsqueeze(1)
        amb_weight = sentiment_probs[:, 2].unsqueeze(1)
        emotion_logits[:, self.positive_idx] *= (1 + pos_weight)
        emotion_logits[:, self.negative_idx] *= (1 + neg_weight)
        emotion_logits[:, self.ambiguous_idx] *= (1 + amb_weight)
        loss = None
        if emotion_labels is not None:
            emotion_loss = self.cb_loss(emotion_logits, emotion_labels.float())
            sentiment_loss = F.binary_cross_entropy_with_logits(
                sentiment_logits, sentiment_labels.float())
            loss = 1.7 * emotion_loss + 0.3 * sentiment_loss
        return {"loss": loss, "emotion_logits": emotion_logits,
                "sentiment_logits": sentiment_logits}


# ── Variant 5: NO Class Balanced Loss (plain BCE instead) ──

class NoCBLossModel(nn.Module):
    def __init__(self, samples_per_class):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.layer_norm = nn.LayerNorm(768)
        self.dropout = nn.Dropout(0.4)
        self.sentiment_head = nn.Linear(768, 3)
        self.emotion_head = nn.Linear(768 + 3, 27)
        self.correlation_layer = LabelCorrelationLayer(27)
        # ❌ NO CB loss — plain BCE
        self.positive_idx = torch.tensor([0,1,4,5,8,13,15,17,18,20,21,23])
        self.negative_idx = torch.tensor([2,3,9,10,11,12,14,16,19,24,25])
        self.ambiguous_idx = torch.tensor([6,7,22,26])

    def forward(self, input_ids, attention_mask,
                emotion_labels=None, sentiment_labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.layer_norm(outputs.pooler_output)
        pooled = self.dropout(pooled)
        sentiment_logits = self.sentiment_head(pooled)
        sentiment_probs = torch.sigmoid(sentiment_logits)
        combined = torch.cat([pooled, sentiment_logits], dim=1)
        emotion_logits = self.emotion_head(combined)
        emotion_logits = self.correlation_layer(emotion_logits)
        pos_weight = sentiment_probs[:, 0].unsqueeze(1)
        neg_weight = sentiment_probs[:, 1].unsqueeze(1)
        amb_weight = sentiment_probs[:, 2].unsqueeze(1)
        emotion_logits[:, self.positive_idx] *= (1 + pos_weight)
        emotion_logits[:, self.negative_idx] *= (1 + neg_weight)
        emotion_logits[:, self.ambiguous_idx] *= (1 + amb_weight)
        loss = None
        if emotion_labels is not None:
            # ❌ plain BCE, no class weighting
            emotion_loss = F.binary_cross_entropy_with_logits(
                emotion_logits, emotion_labels.float())
            sentiment_loss = F.binary_cross_entropy_with_logits(
                sentiment_logits, sentiment_labels.float())
            loss = 1.7 * emotion_loss + 0.3 * sentiment_loss
        return {"loss": loss, "emotion_logits": emotion_logits,
                "sentiment_logits": sentiment_logits}


print("✅ All ablation variants defined")

✅ All ablation variants defined


In [61]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import f1_score
import torch

def run_ablation(model_class, variant_name, samples_per_class):

    print(f"\n{'='*50}")
    print(f"Running: {variant_name}")
    print(f"{'='*50}")

    # Fresh model
    ablation_model = model_class(samples_per_class).to(device)

    args = TrainingArguments(
        output_dir=f"./ablation_{variant_name.replace(' ', '_')}",
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        learning_rate=6e-6,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=10,
        weight_decay=0.02,
        warmup_ratio=0.1,
        load_best_model_at_end=True,
        metric_for_best_model="emotion_macro_f1",
        greater_is_better=True,
        remove_unused_columns=False,
        report_to="none"
    )

    trainer = Trainer(
        model=ablation_model,
        args=args,
        train_dataset=final_train_dataset,
        eval_dataset=validation_dataset,
        compute_metrics=journal_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    # Evaluate on test set
    test_out = trainer.predict(test_dataset)

    emotion_logits = test_out.predictions[0]
    emotion_labels = test_out.label_ids[0]
    sentiment_logits = test_out.predictions[1]
    sentiment_labels = test_out.label_ids[1]

    emotion_probs = torch.sigmoid(torch.tensor(emotion_logits)).numpy()
    emotion_preds = (emotion_probs > 0.5).astype(int)

    sentiment_probs = torch.sigmoid(torch.tensor(sentiment_logits)).numpy()
    sentiment_preds = (sentiment_probs > 0.5).astype(int)

    emotion_macro = f1_score(
        emotion_labels, emotion_preds,
        average="macro", zero_division=0
    )

    sentiment_macro = f1_score(
        sentiment_labels, sentiment_preds,
        average="macro", zero_division=0
    )

    per_class = f1_score(
        emotion_labels, emotion_preds,
        average=None, zero_division=0
    )

    print(f"\nEmotion Macro F1 : {emotion_macro:.4f}")
    print(f"Sentiment Macro F1: {sentiment_macro:.4f}")

    return {
        "variant": variant_name,
        "emotion_macro_f1": emotion_macro,
        "sentiment_macro_f1": sentiment_macro,
        "per_class_f1": per_class
    }

In [ ]:
# ============================================================
# ABLATION STUDY — SAFE VERSION
# Auto cleanup after each variant
# Threshold tuning included
# Disk check before each variant
# ============================================================

from transformers import (
    Trainer, TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.metrics import f1_score
import torch
import numpy as np
import shutil
import glob
import os
import json

# ── Helper: check disk ────────────────────────────────────
def check_disk():
    total = sum(
        os.path.getsize(
            os.path.join(dp, f)
        )
        for dp, dn, files
        in os.walk("/kaggle/working")
        for f in files
    )
    gb   = total / 1e9
    free = 20 - gb
    print(f"  Disk: {gb:.2f} GB used | "
          f"{free:.2f} GB free")
    return free

# ── Helper: cleanup ───────────────────────────────────────
def cleanup(variant_name):
    deleted = 0

    # Delete this variant checkpoints
    folder = (
        f"./ablation_"
        f"{variant_name.replace(' ', '_')}"
    )
    if os.path.exists(folder):
        size = sum(
            os.path.getsize(
                os.path.join(dp, f)
            )
            for dp, dn, files
            in os.walk(folder)
            for f in files
        )
        shutil.rmtree(folder)
        deleted += size
        print(f"  ✅ Deleted: {folder} "
              f"({size/1e6:.0f} MB)")

    # Delete any other ablation folders
    for f in glob.glob("./ablation_*"):
        if os.path.isdir(f):
            size = sum(
                os.path.getsize(
                    os.path.join(dp, fi)
                )
                for dp, dn, files
                in os.walk(f)
                for fi in files
            )
            shutil.rmtree(f)
            deleted += size
            print(f"  ✅ Deleted: {f} "
                  f"({size/1e6:.0f} MB)")

    # Delete zip files
    for z in glob.glob(
        "/kaggle/working/*.zip"
    ):
        size = os.path.getsize(z)
        os.remove(z)
        deleted += size
        print(f"  ✅ Deleted zip "
              f"({size/1e6:.0f} MB)")

    print(f"  Freed: {deleted/1e6:.0f} MB")
    check_disk()

# ── Main ablation function ────────────────────────────────
def run_ablation(model_class, variant_name,
                 samples_per_class):

    print(f"\n{'='*55}")
    print(f"VARIANT: {variant_name}")
    print(f"{'='*55}")

    # Check disk before starting
    print("\nDisk before training:")
    free = check_disk()

    if free < 2.0:
        print("⚠️  Low disk — cleaning first...")
        cleanup(variant_name)

    # ── Fresh model ───────────────────────────────
    ablation_model = model_class(
        samples_per_class
    ).to(device)

    args = TrainingArguments(
        output_dir=(
            f"./ablation_"
            f"{variant_name.replace(' ', '_')}"
        ),
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        learning_rate=6e-6,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=10,
        weight_decay=0.02,
        warmup_ratio=0.1,
        load_best_model_at_end=True,
        metric_for_best_model=
            "emotion_macro_f1",
        greater_is_better=True,
        remove_unused_columns=False,
        report_to="none"
    )

    trainer = Trainer(
        model=ablation_model,
        args=args,
        train_dataset=final_train_dataset,
        eval_dataset=validation_dataset,
        compute_metrics=journal_metrics,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=2
            )
        ]
    )

    trainer.train()

    # ── Test at 0.5 threshold ─────────────────────
    test_out = trainer.predict(test_dataset)

    emotion_logits   = test_out.predictions[0]
    sentiment_logits = test_out.predictions[1]
    emotion_labels   = test_out.label_ids[0]
    sentiment_labels = test_out.label_ids[1]

    emotion_probs = torch.sigmoid(
        torch.tensor(emotion_logits)
    ).numpy()
    sentiment_probs = torch.sigmoid(
        torch.tensor(sentiment_logits)
    ).numpy()

    emotion_preds_05 = (
        emotion_probs > 0.5
    ).astype(int)
    sentiment_preds = (
        sentiment_probs > 0.5
    ).astype(int)

    emotion_macro_05 = f1_score(
        emotion_labels, emotion_preds_05,
        average="macro", zero_division=0
    )
    sentiment_macro = f1_score(
        sentiment_labels, sentiment_preds,
        average="macro", zero_division=0
    )
    per_class_05 = f1_score(
        emotion_labels, emotion_preds_05,
        average=None, zero_division=0
    )

    # ── Threshold tuning ──────────────────────────
    print("\nTuning thresholds...")

    val_out    = trainer.predict(
        validation_dataset
    )
    val_logits = val_out.predictions[0]
    val_labels = val_out.label_ids[0]
    val_probs  = torch.sigmoid(
        torch.tensor(val_logits)
    ).numpy()

    best_thresholds_abl = []

    for i in range(27):
        best_f1 = 0
        best_t  = 0.5

        for t in np.arange(0.1, 0.9, 0.05):
            preds = (
                val_probs[:, i] > t
            ).astype(int)
            f1 = f1_score(
                val_labels[:, i],
                preds,
                zero_division=0
            )
            if f1 > best_f1:
                best_f1 = f1
                best_t  = t

        best_thresholds_abl.append(best_t)

    best_thresholds_abl = np.array(
        best_thresholds_abl
    )

    # ── Test at tuned threshold ───────────────────
    emotion_preds_tuned = (
        emotion_probs > best_thresholds_abl
    ).astype(int)

    emotion_macro_tuned = f1_score(
        emotion_labels,
        emotion_preds_tuned,
        average="macro", zero_division=0
    )
    per_class_tuned = f1_score(
        emotion_labels,
        emotion_preds_tuned,
        average=None, zero_division=0
    )



    #SAVE RESULTS
    result = {
    "variant":             variant_name,
    "emotion_macro_f1":    emotion_macro_05,
    "emotion_macro_tuned": emotion_macro_tuned,
    "sentiment_macro_f1":  sentiment_macro,
    "per_class_f1":        per_class_05.tolist(),
    "per_class_tuned":     per_class_tuned.tolist()
}

    # Save immediately after each variant
    existing = []
    json_path = "/kaggle/working/ablation_results.json"

    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            existing = json.load(f)

    existing.append(result)

    with open(json_path, "w") as f:
        json.dump(existing, f, indent=2)

    print(f"\n  ✅ Result saved to "
          f"ablation_results.json")

    # ── Cleanup after variant ─────────────────────
    print(f"\nCleaning up disk...")
    cleanup(variant_name)

    return result


# ============================================================
# RUN ALL 4 VARIANTS
# ============================================================

ablation_results = []

variants = [
    (NoCorrelationModel,
     "No Label Correlation"),
    (NoModulationModel,
     "No Sentiment Modulation"),
    (NoHierarchyModel,
     "No Hierarchical Injection"),
    (NoCBLossModel,
     "No Class-Balanced Loss"),
]

for model_class, name in variants:
    result = run_ablation(
        model_class,
        name,
        emotion_counts_balanced
    )
    ablation_results.append(result)
    print(f"\n✅ {name} complete")
    print(f"   Saved to JSON ✅")

print("\n" + "=" * 55)
print("✅ ALL ABLATION EXPERIMENTS COMPLETE")
print("=" * 55)

# ============================================================
# ABLATION SUMMARY TABLE — 4 variants only

# ============================================================

import pandas as pd

rows = []

for r in ablation_results:
    rows.append({
        "Variant":      r["variant"],
        "F1 (τ=0.5)":  round(
            r["emotion_macro_f1"], 4
        ),
        "F1 (Tuned)":   round(
            r["emotion_macro_tuned"], 4
        ),
        "Sentiment F1": round(
            r["sentiment_macro_f1"], 4
        )
    })

df = pd.DataFrame(rows)

print("\n===== ABLATION STUDY RESULTS =====\n")
print(df.to_string(index=False))

# Save
df.to_csv(
    "/kaggle/working/ablation_summary.csv",
    index=False
)
print("\n✅ Saved: ablation_summary.csv")



VARIANT: No Label Correlation

Disk before training:
  Disk: 0.00 GB used | 20.00 GB free


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Emotion Macro F1,Sentiment Macro F1
1,0.595677,0.266010,0.191739,0.786863
2,0.284190,0.223039,0.446654,0.799055
3,0.240053,0.212978,0.505686,0.804222
4,0.215292,0.215877,0.521331,0.806770
5,0.196785,0.216892,0.528885,0.807421
6,0.182665,0.220280,0.527967,0.804804
7,0.170394,0.222846,0.534881,0.800127
8,0.161511,0.229494,0.530926,0.799057
9,0.154990,0.230847,0.536149,0.800951
10,0.150789,0.232082,0.538196,0.803544



Tuning thresholds...


...


Tuning thresholds...



  ✅ Result saved to ablation_results.json

Cleaning up disk...
  ✅ Deleted: ./ablation_No_Sentiment_Modulation (13143 MB)
  Freed: 13143 MB
  Disk: 0.00 GB used | 20.00 GB free

✅ No Sentiment Modulation complete
   Saved to JSON ✅

VARIANT: No Hierarchical Injection

Disk before training:
  Disk: 0.00 GB used | 20.00 GB free


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Emotion Macro F1,Sentiment Macro F1
1,0.634103,0.280222,0.163991,0.778883
2,0.300591,0.230831,0.436489,0.793709
3,0.250236,0.216875,0.498926,0.800209


In [65]:
import json, os

path = "/kaggle/working/ablation_results.json"

if os.path.exists(path):
    with open(path) as f:
        data = json.load(f)

    print("Saved variants:\n")
    for r in data:
        print(r["variant"],
              "| F1(0.5):", round(r["emotion_macro_f1"],4),
              "| F1(tuned):", round(r["emotion_macro_tuned"],4),
              "| Sentiment F1:", round(r["sentiment_macro_f1"],4))
else:
    print("No results saved yet")

Saved variants:

No Label Correlation | F1(0.5): 0.5343 | F1(tuned): 0.5553 | Sentiment F1: 0.8119
No Sentiment Modulation | F1(0.5): 0.5415 | F1(tuned): 0.5499 | Sentiment F1: 0.8117
No Hierarchical Injection | F1(0.5): 0.5262 | F1(tuned): 0.5493 | Sentiment F1: 0.8185


In [66]:
result = run_ablation(
    NoCBLossModel,
    "No Class-Balanced Loss",
    emotion_counts_balanced
)

print(result)


VARIANT: No Class-Balanced Loss

Disk before training:
  Disk: 3.94 GB used | 16.06 GB free


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Emotion Macro F1,Sentiment Macro F1
1,0.678035,0.310269,0.191207,0.775432
2,0.324247,0.253407,0.429145,0.794852
3,0.270844,0.240091,0.485817,0.801200
4,0.243019,0.240503,0.515186,0.804727
5,0.222690,0.241496,0.524980,0.807991
6,0.207822,0.243401,0.522009,0.807732
7,0.195643,0.245628,0.530808,0.807431
8,0.185733,0.251618,0.534982,0.807373
9,0.179366,0.253105,0.538973,0.807718
10,0.174757,0.253472,0.543243,0.806496



Tuning thresholds...



  ✅ Result saved to ablation_results.json

Cleaning up disk...
  ✅ Deleted: ./ablation_No_Class-Balanced_Loss (13143 MB)
  Freed: 13143 MB
  Disk: 0.00 GB used | 20.00 GB free
{'variant': 'No Class-Balanced Loss', 'emotion_macro_f1': 0.5412021123690366, 'emotion_macro_tuned': 0.5651623470382531, 'sentiment_macro_f1': 0.8155514362869063, 'per_class_f1': [0.752, 0.8544776119402985, 0.5214899713467048, 0.3835051546391753, 0.5060658578856152, 0.5172413793103449, 0.4534412955465587, 0.6924493554327809, 0.5538461538461539, 0.233502538071066, 0.49888641425389757, 0.5360824742268041, 0.4307692307692308, 0.47058823529411764, 0.7162162162162162, 0.9253294289897511, 0.3333333333333333, 0.6176470588235294, 0.8042553191489362, 0.3333333333333333, 0.5749235474006116, 0.36363636363636365, 0.2857142857142857, 0.4375, 0.704, 0.5660377358490566, 0.5461847389558233], 'per_class_tuned': [0.752, 0.8617594254937163, 0.5625, 0.45098039215686275, 0.5340453938584779, 0.5109489051094891, 0.49710982658959535, 0

In [67]:
import json
import pandas as pd

# Load ablation results
with open("/kaggle/working/ablation_results.json") as f:
    data = json.load(f)

# Full model results
full_model = {
    "variant": "Full Model",
    "emotion_macro_f1": 0.5527580750022073,
    "emotion_macro_tuned": 0.5604443893948465,
    "sentiment_macro_f1": 0.8113347687154612
}

# Insert full model at top
data.insert(0, full_model)

rows = []
for r in data:
    rows.append({
        "Variant": r["variant"],
        "F1 (τ=0.5)": round(r["emotion_macro_f1"],4),
        "F1 (Tuned)": round(r["emotion_macro_tuned"],4),
        "Sentiment F1": round(r["sentiment_macro_f1"],4)
    })

df = pd.DataFrame(rows)

print("\n===== FINAL ABLATION TABLE =====\n")
print(df.to_string(index=False))

# Save for paper
df.to_csv("/kaggle/working/ablation_summary_final.csv", index=False)

print("\nSaved: ablation_summary_final.csv")


===== FINAL ABLATION TABLE =====

                  Variant  F1 (τ=0.5)  F1 (Tuned)  Sentiment F1
               Full Model      0.5528      0.5604        0.8113
     No Label Correlation      0.5343      0.5553        0.8119
  No Sentiment Modulation      0.5415      0.5499        0.8117
No Hierarchical Injection      0.5262      0.5493        0.8185
   No Class-Balanced Loss      0.5412      0.5652        0.8156

Saved: ablation_summary_final.csv


In [ ]:
import json
import pandas as pd

# Load ablation results
with open("/kaggle/working/ablation_results.json") as f:
    data = json.load(f)
# Full model results
full_model = {
    "variant": "Full Model",
    "emotion_macro_f1": 0.5527580750022073,
    "emotion_macro_tuned": 0.5604443893948465,
    "sentiment_macro_f1": 0.8113347687154612
}

# Insert full model at top
data.insert(0, full_model)

rows = []
for r in data:
    rows.append({
        "Variant": r["variant"],
        "F1 (τ=0.5)": round(r["emotion_macro_f1"],2),
        "F1 (Tuned)": round(r["emotion_macro_tuned"],2),
        "Sentiment F1": round(r["sentiment_macro_f1"],2)
    })

df = pd.DataFrame(rows)

print("\n===== FINAL ABLATION TABLE =====\n")
print(df.to_string(index=False))

# Save for paper
df.to_csv("/kaggle/working/ablation_summary_final.csv", index=False)

print("\nSaved: ablation_summary_final.csv")

In [71]:
FULL_MODEL = 0.5604  # your tuned full model F1

df["Performance Drop"] = FULL_MODEL - df["F1 (Tuned)"]

print(df)

                     Variant  F1 (τ=0.5)  F1 (Tuned)  Sentiment F1  \
0                 Full Model      0.5528      0.5604        0.8113   
1       No Label Correlation      0.5343      0.5553        0.8119   
2    No Sentiment Modulation      0.5415      0.5499        0.8117   
3  No Hierarchical Injection      0.5262      0.5493        0.8185   
4     No Class-Balanced Loss      0.5412      0.5652        0.8156   

   Performance Drop  
0            0.0000  
1            0.0051  
2            0.0105  
3            0.0111  
4           -0.0048  


In [74]:
# Focus on the 5 rarest classes
rare_classes = ["grief", "nervousness", "pride", "relief", "embarrassment"]
rare_indices = [EMOTION_NAMES_NO_NEUTRAL.index(c) for c in rare_classes]

print("\n===== RARE CLASS F1 PER VARIANT =====\n")

rare_df = pd.DataFrame(
    {r["variant"]: [round(r["per_class_f1"][i], 4) for i in rare_indices]
     for r in ablation_results},
    index=rare_classes
).T

print(rare_df.to_string())


===== RARE CLASS F1 PER VARIANT =====

                            grief  nervousness   pride  relief  embarrassment
No Label Correlation       0.2353       0.2963  0.3871  0.3889         0.4507
No Sentiment Modulation    0.4286       0.3571  0.3871  0.4118         0.4706
No Hierarchical Injection  0.1176       0.3667  0.3871  0.4103         0.4348


In [75]:
import json
import pandas as pd

# Load saved ablation results
with open("/kaggle/working/ablation_results.json", "r") as f:
    ablation_results = json.load(f)

# Rare classes
rare_classes = ["grief", "nervousness", "pride", "relief", "embarrassment"]
rare_indices = [EMOTION_NAMES_NO_NEUTRAL.index(c) for c in rare_classes]

print("\n===== RARE CLASS F1 PER VARIANT =====\n")

rare_df = pd.DataFrame(
    {
        r["variant"]: [round(r["per_class_f1"][i], 4) for i in rare_indices]
        for r in ablation_results
    },
    index=rare_classes
).T

print(rare_df.to_string())


===== RARE CLASS F1 PER VARIANT =====

                            grief  nervousness   pride  relief  embarrassment
No Label Correlation       0.2353       0.2963  0.3871  0.3889         0.4507
No Sentiment Modulation    0.4286       0.3571  0.3871  0.4118         0.4706
No Hierarchical Injection  0.1176       0.3667  0.3871  0.4103         0.4348
No Class-Balanced Loss     0.3333       0.3333  0.3636  0.4375         0.4308


In [79]:
import json
import pandas as pd

# Load saved ablation results
with open("/kaggle/working/ablation_results.json", "r") as f:
    ablation_results = json.load(f)




print("\n===== PER EMOTION F1 PER VARIANT =====\n")

emotion_df_T = emotion_df.T
print(emotion_df_T.to_string())



===== PER EMOTION F1 PER VARIANT =====

                No Label Correlation  No Sentiment Modulation  No Hierarchical Injection  No Class-Balanced Loss
admiration                    0.7259                   0.7226                     0.7037                  0.7520
amusement                     0.8598                   0.8692                     0.8477                  0.8545
anger                         0.5371                   0.5341                     0.5378                  0.5215
annoyance                     0.3243                   0.2763                     0.2368                  0.3835
approval                      0.5087                   0.4815                     0.4879                  0.5061
caring                        0.5169                   0.4793                     0.5047                  0.5172
confusion                     0.4739                   0.4681                     0.5098                  0.4534
curiosity                     0.6933                   